# Model: will the runner be safe?

When a single goes to the outfield with a runner on second, the third-base coach has to decide: **send** the runner home or **hold** him at third. I call each of these plays an *opportunity*.

In this notebook I build a model that looks at an opportunity and estimates **P(safe)**, the chance the runner scores if he is sent. I train it on 2022–2025 sends, where I know whether the runner was safe or out, and then use it to predict every 2026 opportunity. The model never sees 2026 while it learns, so its 2026 predictions are a fair test.

## Setup

`FEATURES` are the facts about each play that I let the model see:

| Feature | What it means |
|---|---|
| `launch_speed` | Exit velocity: how hard the ball was hit, in mph |
| `launch_angle` | How steeply the ball left the bat, in degrees (negative means down into the ground) |
| `spray_angle` | Direction: about −30 is left field, 0 is straight up the middle, +30 is right field |
| `outs_when_up` | Outs before the play: 0, 1 or 2 |
| `runner_speed` | The runner's sprint speed from Baseball Savant, in feet per second |
| `fielder_arm` | The outfielder's arm strength from Baseball Savant, in mph |
| `bb_type` | Ground ball, line drive, fly ball or popup |

If a player has no Savant number for a season, or too few runs or throws to measure, I use the middle (median) value for that season instead.

In [ ]:
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier

TEST_SEASON = 2026
FEATURES = ["launch_speed", "launch_angle", "spray_angle", "outs_when_up",
            "runner_speed", "fielder_arm", "bb_type"]

## Load the plays

Each row of `opportunities.csv` is one opportunity. A few plays are missing exit velocity, launch angle or direction, so I keep only the `complete` ones.

The model needs numbers, but `bb_type` is a label. I use `get_dummies` to turn each label into its own yes/no column of 1s and 0s, like `bb_type_line_drive`.

- `train`: 2022–2025 plays where the runner was sent. These are the only plays where I know the answer, safe or out.
- `test`: every 2026 play, sent or held.

In [ ]:
plays = pd.read_csv("../opportunities.csv")
plays = plays[plays.complete]

X = pd.get_dummies(plays[FEATURES], columns=["bb_type"], dtype=float)

train = plays.sent & (plays.season < TEST_SEASON)
test = plays.season == TEST_SEASON

## Train the model and predict 2026

I chose a *gradient boosting* model. It builds a long chain of small decision trees. Each tree is a couple of yes/no questions, like "Was the ball hit harder than 95 mph?" or "Is the runner faster than 28 ft/s?", that end in a guess. Each new tree focuses on fixing the mistakes of the trees before it, and the final answer combines all of them.

I set `max_depth=2` to limit each tree to two questions in a row. Outs are rare, only about 350 in four seasons, so bigger trees would start memorizing flukes instead of learning real patterns.

I set `early_stopping=False` so the model learns from every training send. Without it, scikit-learn quietly sets aside 10% of the plays whenever there are more than 10,000. I left everything else, including the number of trees (100), at scikit-learn's standard settings.

`predict_proba` gives two chances for each play, out and safe. I keep `[:, 1]`, the chance of safe.

In [ ]:
model = HistGradientBoostingClassifier(max_depth=2, early_stopping=False, random_state=0)
model.fit(X[train], plays.safe[train].astype(int))
plays.loc[test, "p_safe"] = model.predict_proba(X[test])[:, 1]

## Save

Every 2026 play now has `p_safe`. I leave plays before 2026 blank, because the model learned from them.

In [ ]:
plays.to_csv("../scored.csv", index=False)